# LungCT-NET-Inspired Three-Class Classification

This notebook adapts the transfer-learning and ensemble ideas described for **LungCT-NET** to this project's three classes: **Benign, Malignant, and Normal**. It uses the existing, already split and augmented dataset without changing its files, creating new splits, or applying augmentation.

The notebook trains VGG16, VGG19, MobileNetV2, and EfficientNetB0, then fits a leakage-conscious stacking ensemble. It saves results in a new timestamped folder on Google Drive.

> **Important:** This is an experimental image-classification workflow, not a medical device or clinical decision tool. Review dataset counts and resource settings before a full run.


## 1. Runtime setup and configuration

Run this notebook in Google Colab with Google Drive access. ImageNet weights may need to download the first time. Training the CNNs and generating five-fold OOF predictions is computationally expensive.

The exact VGG Flatten → Dense(4096) head is available, but has very high memory demand. For a practical first run, the default uses global average pooling (GAP) for VGG; set `VGG_HEAD_MODE = "paper_flatten"` to use the paper-described VGG head if GPU memory permits. This head choice is recorded in the experiment configuration.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import os
import platform
import random
import sys
import time
import warnings
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

DEFAULT_DRIVE_ROOT = Path("/content/drive")
if IS_COLAB:
    if (DEFAULT_DRIVE_ROOT / "MyDrive").is_dir():
        DRIVE_ROOT = DEFAULT_DRIVE_ROOT
    else:
        mountpoint = DEFAULT_DRIVE_ROOT
        if mountpoint.exists() and any(mountpoint.iterdir()):
            mountpoint = Path("/content/gdrive")
            if mountpoint.exists() and any(mountpoint.iterdir()):
                raise RuntimeError(
                    "Both /content/drive and /content/gdrive contain files, so Google Drive "
                    "cannot be mounted safely. Choose an empty mountpoint and update this cell."
                )
        mountpoint.mkdir(parents=True, exist_ok=True)
        drive.mount(str(mountpoint))
        DRIVE_ROOT = mountpoint
else:
    DRIVE_ROOT = DEFAULT_DRIVE_ROOT

DATASET_ROOT = DRIVE_ROOT / "MyDrive/Dataset/Lung_Cancer_Preprocessed_Dataset"
RESULTS_ROOT = DRIVE_ROOT / "MyDrive/Lung_Cancer_Classification_Results/LungCT_NET_Three_Class_Classification"
CLASS_NAMES = ["Benign", "Malignant", "Normal"]
EXPECTED_COUNTS = {
    "train": {"Benign": 735, "Malignant": 735, "Normal": 735},
    "validation": {"Benign": 70, "Malignant": 157, "Normal": 62},
    "test": {"Benign": 71, "Malignant": 158, "Normal": 63},
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 2
EPOCHS = 100
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 10
OOF_FOLDS = 5
PIPELINE_CACHE = False
VGG_HEAD_MODE = "gap"  # "gap" (recommended) or "paper_flatten"
ENABLE_GRAD_CAM = False
ENABLE_SHAP = False
SHAP_BACKGROUND_SIZE = 2
SHAP_EXPLAIN_SIZE = 1

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR = RESULTS_ROOT / f"run_{RUN_ID}"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
PLOT_DIR = RUN_DIR / "plots"
HISTORY_DIR = RUN_DIR / "histories"
for folder in (CHECKPOINT_DIR, PLOT_DIR, HISTORY_DIR):
    folder.mkdir(parents=True, exist_ok=False)

print("Run folder:", RUN_DIR)
print("Dataset root:", DATASET_ROOT)
print("Classes (fixed label order):", CLASS_NAMES)
print("VGG head mode:", VGG_HEAD_MODE)
print("Five-fold OOF will train 20 additional CNNs (4 architectures x 5 folds).")


## 2. Reproducibility and environment record

Seeds and TensorFlow deterministic operations are enabled where supported. GPU kernels, library versions, hardware, and external ImageNet downloads can still prevent bit-for-bit reproducibility. The notebook records the runtime and configuration rather than claiming exact reproducibility.


In [2]:
import tensorflow as tf
from sklearn import __version__ as sklearn_version

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISM_ENABLED = True
except (AttributeError, RuntimeError) as exc:
    DETERMINISM_ENABLED = False
    print("TensorFlow deterministic operations unavailable:", exc)

GPU_DEVICES = tf.config.list_physical_devices("GPU")
for device in GPU_DEVICES:
    try:
        tf.config.experimental.set_memory_growth(device, True)
    except RuntimeError as exc:
        print(f"Could not enable memory growth for {device.name}: {exc}")

def package_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed / distribution name unavailable"

ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "keras": getattr(tf.keras, "__version__", package_version("keras")),
    "numpy": np.__version__,
    "scikit_learn": sklearn_version,
    "pandas": pd.__version__,
    "matplotlib": package_version("matplotlib"),
    "pillow": package_version("Pillow"),
    "gpu_devices": [device.name for device in GPU_DEVICES],
    "deterministic_ops_enabled": DETERMINISM_ENABLED,
}
print(json.dumps(ENVIRONMENT, indent=2))


## 3. Inspect the existing dataset before training

The check below counts supported image files in each class folder and compares them with the expected table. It reports differences without moving, removing, or modifying data. The loader uses the observed files, while the labels always use the fixed class order above. If a folder is missing or class folders differ, execution stops rather than silently remapping labels.


In [3]:
# Drive is mounted in the setup cell, before results folders are created.
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_ROOT}. Confirm that the Drive mounted at "
        f"{DRIVE_ROOT} contains Lung_Cancer_Preprocessed_Dataset at the configured path, "
        "or update DATASET_ROOT in the setup cell."
    )

def image_files_in(folder):
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

split_files = {}
observed_rows = []
extension_rows = []

for split_name in ("train", "validation", "test"):
    split_dir = DATASET_ROOT / split_name
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Expected split folder is missing: {split_dir}")

    observed_class_dirs = sorted(path.name for path in split_dir.iterdir() if path.is_dir())
    if observed_class_dirs != sorted(CLASS_NAMES):
        raise ValueError(
            f"{split_name} class folders are {observed_class_dirs}; "
            f"expected exactly {sorted(CLASS_NAMES)}."
        )

    split_files[split_name] = {}
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        files = image_files_in(class_dir)
        split_files[split_name][class_name] = files
        observed_rows.append({
            "split": split_name,
            "class": class_name,
            "expected": EXPECTED_COUNTS[split_name][class_name],
            "observed": len(files),
            "difference": len(files) - EXPECTED_COUNTS[split_name][class_name],
        })
        extension_counts = {}
        for path in files:
            extension_counts[path.suffix.lower()] = extension_counts.get(path.suffix.lower(), 0) + 1
        extension_rows.append({
            "split": split_name,
            "class": class_name,
            "extensions": extension_counts,
        })

counts_df = pd.DataFrame(observed_rows)
display(counts_df)
print("Observed image extensions by split and class:")
display(pd.DataFrame(extension_rows))

if (counts_df["difference"] != 0).any():
    print("WARNING: observed image counts differ from the requested expected counts above.")
    print("No files were changed, and no files were added, removed, or re-split.")
else:
    print("All expected split/class image counts match.")

if (counts_df["observed"] == 0).any():
    raise ValueError("At least one class folder has no supported images; inspect the count table.")

def arrays_for_split(split_name):
    paths, labels = [], []
    for label, class_name in enumerate(CLASS_NAMES):
        for path in split_files[split_name][class_name]:
            paths.append(str(path))
            labels.append(label)
    return np.asarray(paths, dtype=str), np.asarray(labels, dtype=np.int32)

train_paths, y_train = arrays_for_split("train")
val_paths, y_val = arrays_for_split("validation")
test_paths, y_test = arrays_for_split("test")

print(f"Loaded path/label counts: train={len(y_train)}, validation={len(y_val)}, test={len(y_test)}")
print("Integer label map:", {name: index for index, name in enumerate(CLASS_NAMES)})


## 4. Input pipeline and architecture-specific preprocessing

Images are decoded as RGB and resized only for model input; the files on Drive remain untouched. No augmentation is applied. Each model gets the preprocessing convention expected by its ImageNet weights: VGG subtracts channel means and converts RGB to BGR, MobileNetV2 scales pixels to `[-1, 1]`, and Keras EfficientNetB0 includes input rescaling internally (its `preprocess_input` is effectively a pass-through for 0–255 pixels).


In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg19_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_b0_preprocess

PREPROCESSORS = {
    "VGG16": vgg16_preprocess,
    "VGG19": vgg19_preprocess,
    "MobileNetV2": mobilenet_v2_preprocess,
    "EfficientNetB0": efficientnet_b0_preprocess,
}
MODEL_NAMES = list(PREPROCESSORS)

def make_dataset(paths, labels, model_name, training=False, seed=SEED):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    preprocess = PREPROCESSORS[model_name]

    def decode_and_prepare(path, label):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
        image = tf.cast(image, tf.float32)  # Keep the 0..255 range before model-specific preprocessing.
        image = preprocess(image)
        return image, label

    dataset = dataset.map(decode_and_prepare, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    if PIPELINE_CACHE:
        dataset = dataset.cache()
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=False)
    return dataset.prefetch(tf.data.AUTOTUNE)


## 5. Build the four transfer-learning CNNs

Every convolutional base starts frozen and uses ImageNet weights with `include_top=False`. All heads produce three softmax probabilities. The VGG head mode is configurable; exact paper-style Flatten heads can require several gigabytes due to their large dense layers. For the default GAP variant, model capacity and memory use are substantially lower.


In [ ]:
def build_model(model_name, vgg_head_mode=VGG_HEAD_MODE):
    input_shape = (*IMAGE_SIZE, 3)
    input_tensor = tf.keras.Input(shape=input_shape, name="image")

    constructors = {
        "VGG16": tf.keras.applications.VGG16,
        "VGG19": tf.keras.applications.VGG19,
        "MobileNetV2": tf.keras.applications.MobileNetV2,
        "EfficientNetB0": tf.keras.applications.EfficientNetB0,
    }
    if model_name not in constructors:
        raise ValueError(f"Unknown model: {model_name}")

    backbone = constructors[model_name](
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        name="feature_extractor",
    )
    backbone.trainable = False
    features = backbone(input_tensor, training=False)

    if model_name in ("VGG16", "VGG19") and vgg_head_mode == "paper_flatten":
        x = tf.keras.layers.Flatten(name="flatten")(features)
        x = tf.keras.layers.Dense(4096, activation="relu", name="dense_4096_a")(x)
        x = tf.keras.layers.Dropout(0.5, name="dropout_a")(x)
        x = tf.keras.layers.Dense(4096, activation="relu", name="dense_4096_b")(x)
        x = tf.keras.layers.Dropout(0.5, name="dropout_b")(x)
    elif model_name in ("VGG16", "VGG19"):
        x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling2d")(features)
        x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
        x = tf.keras.layers.Dropout(0.5, name="dropout")(x)
    elif model_name in ("MobileNetV2", "EfficientNetB0"):
        x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling2d")(features)
        x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
    else:
        raise ValueError(f"Unsupported VGG head mode: {vgg_head_mode}")

    output = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="class_probabilities")(x)
    model = tf.keras.Model(input_tensor, output, name=f"{model_name}_three_class")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model

for name in MODEL_NAMES:
    preview = build_model(name)
    print(f"{name}: {preview.count_params():,} parameters; backbone frozen.")
    del preview
    tf.keras.backend.clear_session()


## 6. Train on train; select checkpoints using validation

The training split is used for gradient updates. The validation split is used only for early stopping and best-checkpoint selection. The test split is not loaded by `fit`. Loss and accuracy are saved per epoch, along with learning rate and wall-clock training time.


In [ ]:
class LearningRateRecorder(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.values = []

    def on_epoch_end(self, epoch, logs=None):
        learning_rate = self.model.optimizer.learning_rate
        self.values.append(float(tf.keras.backend.get_value(learning_rate)))

def train_full_model(model_name):
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(model_name)
    train_ds = make_dataset(train_paths, y_train, model_name, training=True, seed=SEED)
    validation_ds = make_dataset(val_paths, y_val, model_name, training=False)
    checkpoint_path = CHECKPOINT_DIR / f"{model_name}_best.keras"
    lr_recorder = LearningRateRecorder()
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        lr_recorder,
    ]

    started = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    elapsed_seconds = time.perf_counter() - started
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    # Persist the best validation checkpoint, not a later non-best in-memory state.
    history_payload = {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    }
    history_payload["learning_rate"] = lr_recorder.values
    history_payload["training_time_seconds"] = elapsed_seconds
    history_payload["best_epoch"] = best_epoch
    with (HISTORY_DIR / f"{model_name}_history.json").open("w") as file:
        json.dump(history_payload, file, indent=2)
    del model
    tf.keras.backend.clear_session()
    return checkpoint_path, history_payload


In [ ]:
model_paths = {}
histories = {}
training_times = {}
best_epochs = {}

for model_name in MODEL_NAMES:
    print(f"\n===== Training {model_name} =====")
    model_paths[model_name], histories[model_name] = train_full_model(model_name)
    training_times[model_name] = histories[model_name]["training_time_seconds"]
    best_epochs[model_name] = histories[model_name]["best_epoch"]
    print(
        f"{model_name}: best epoch={best_epochs[model_name]}, "
        f"training time={training_times[model_name] / 60:.1f} minutes"
    )


## 7. Training curves

For each CNN, compare training and validation accuracy and loss, and inspect the learning-rate trace. A growing train/validation gap (especially falling train loss with worsening validation loss) suggests overfitting. Persistently poor training and validation performance can indicate underfitting, an optimization issue, or a data/input problem. Curves alone do not establish clinical usefulness.


In [ ]:
for model_name in MODEL_NAMES:
    history = histories[model_name]
    epochs_seen = np.arange(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs_seen, history["accuracy"], label="Train")
    axes[0].plot(epochs_seen, history["val_accuracy"], label="Validation")
    axes[0].set(title=f"{model_name}: accuracy", xlabel="Epoch", ylabel="Accuracy")
    axes[0].legend()

    axes[1].plot(epochs_seen, history["loss"], label="Train")
    axes[1].plot(epochs_seen, history["val_loss"], label="Validation")
    axes[1].set(title=f"{model_name}: loss", xlabel="Epoch", ylabel="Sparse categorical cross-entropy")
    axes[1].legend()

    axes[2].plot(epochs_seen, history["learning_rate"])
    axes[2].set(title=f"{model_name}: learning rate", xlabel="Epoch", ylabel="Learning rate")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{model_name}_training_curves.png", dpi=160, bbox_inches="tight")
    plt.show()


## 8. Generate leakage-safe CNN out-of-fold (OOF) training features

The stacker must not learn from predictions made on the same images used to fit those CNNs. For each of five stratified folds, each architecture is freshly initialized with ImageNet weights and trained only on the other four training folds. Predictions are written only for that fold's held-out training images. The fold models use fixed epoch counts chosen from each architecture's best full-training validation epoch; they do not train on the validation or test images.

This is the most expensive stage: it fits 20 extra CNNs. Set `OOF_FOLDS` lower only for a quick experiment (at least 2); fewer folds make OOF features noisier. No validation or test examples enter the OOF feature matrix.


In [ ]:
from sklearn.model_selection import StratifiedKFold

if OOF_FOLDS < 2:
    raise ValueError("OOF_FOLDS must be at least 2.")
if np.min(np.bincount(y_train)) < OOF_FOLDS:
    raise ValueError("Every training class must have at least OOF_FOLDS images.")

oof_feature_names = [
    f"{model_name}__{class_name}"
    for model_name in MODEL_NAMES
    for class_name in CLASS_NAMES
]
oof_probabilities = np.full((len(y_train), len(MODEL_NAMES), len(CLASS_NAMES)), np.nan, dtype=np.float32)
fold_assignments = np.full(len(y_train), -1, dtype=np.int32)
cv = StratifiedKFold(n_splits=OOF_FOLDS, shuffle=True, random_state=SEED)

for fold_number, (fold_train_indices, fold_holdout_indices) in enumerate(cv.split(train_paths, y_train), start=0):
    fold_assignments[fold_holdout_indices] = fold_number
    print(
        f"OOF fold {fold_number + 1}/{OOF_FOLDS}: "
        f"fit={len(fold_train_indices)}, predict-held-out={len(fold_holdout_indices)}"
    )
    for model_index, model_name in enumerate(MODEL_NAMES):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED + 1000 * (fold_number + 1) + model_index)
        fold_model = build_model(model_name)
        fold_train_ds = make_dataset(
            train_paths[fold_train_indices],
            y_train[fold_train_indices],
            model_name,
            training=True,
            seed=SEED + 1000 * (fold_number + 1) + model_index,
        )
        holdout_ds = make_dataset(
            train_paths[fold_holdout_indices],
            y_train[fold_holdout_indices],
            model_name,
            training=False,
        )
        fixed_epochs = max(1, int(best_epochs[model_name]))
        fold_model.fit(fold_train_ds, epochs=fixed_epochs, verbose=0)
        oof_probabilities[fold_holdout_indices, model_index, :] = fold_model.predict(
            holdout_ds, verbose=0
        )
        del fold_model, fold_train_ds, holdout_ds
        tf.keras.backend.clear_session()

if np.isnan(oof_probabilities).any() or (fold_assignments < 0).any():
    raise RuntimeError("OOF generation did not produce exactly one prediction per training sample.")

oof_matrix = oof_probabilities.reshape(len(y_train), -1)
oof_df = pd.DataFrame(oof_matrix, columns=oof_feature_names)
oof_df.insert(0, "label", y_train)
oof_df.insert(0, "fold", fold_assignments)
oof_df.to_csv(RUN_DIR / "training_oof_features.csv", index=False)
np.savez_compressed(
    RUN_DIR / "training_oof_features.npz",
    probabilities=oof_probabilities,
    labels=y_train,
    fold_assignments=fold_assignments,
)
print("OOF feature matrix:", oof_matrix.shape)


## 9. Fit the tree base classifiers and logistic-regression meta-classifier

Decision Tree, Random Forest, and Extra Trees are trained on the training-only CNN OOF probability features. `StackingClassifier` then creates cross-validated predicted probabilities from these tree models to train Logistic Regression as the final meta-classifier. Finally, the tree models are refit on all OOF training rows. Validation and test rows are not used in `fit`.

The provided paper description does not specify enough detail to verify an identical stacking procedure. This is a transparent, standard sklearn implementation of the requested two-level design, not a claim of exact reproduction.


In [ ]:
import joblib
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

tree_estimators = [
    ("decision_tree", DecisionTreeClassifier(
        max_depth=None, random_state=SEED, class_weight="balanced"
    )),
    ("random_forest", RandomForestClassifier(
        n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced"
    )),
    ("extra_trees", ExtraTreesClassifier(
        n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced"
    )),
]
meta_cv = StratifiedKFold(n_splits=OOF_FOLDS, shuffle=True, random_state=SEED + 1)
stacking_model = StackingClassifier(
    estimators=tree_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=SEED),
    cv=meta_cv,
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=1,
)
stacking_model.fit(oof_matrix, y_train)
joblib.dump(stacking_model, RUN_DIR / "stacking_model.joblib")
print("Stacker fit on OOF training features only.")


## 10. Prepare untouched test predictions

The four full-training CNN checkpoints are now used for inference. The test images are not used to fit or select CNNs or stackers. This cell creates a fixed-width 12-column feature matrix (four models × three class probabilities) for stack inference.


In [ ]:
def predict_probabilities(model, paths, labels, model_name):
    dataset = make_dataset(paths, labels, model_name, training=False)
    probabilities = model.predict(dataset, verbose=0)
    if probabilities.shape != (len(paths), len(CLASS_NAMES)):
        raise ValueError(f"Unexpected output shape from {model_name}: {probabilities.shape}")
    return probabilities

cnn_test_probabilities = {}
for model_name in MODEL_NAMES:
    cnn_model = tf.keras.models.load_model(model_paths[model_name])
    cnn_test_probabilities[model_name] = predict_probabilities(
        cnn_model, test_paths, y_test, model_name
    )
    del cnn_model
    tf.keras.backend.clear_session()

test_feature_matrix = np.concatenate(
    [cnn_test_probabilities[name] for name in MODEL_NAMES],
    axis=1,
)
stack_test_probabilities = stacking_model.predict_proba(test_feature_matrix)
if list(stacking_model.classes_) != list(range(len(CLASS_NAMES))):
    raise ValueError(f"Unexpected stacker class order: {stacking_model.classes_}")

all_test_probabilities = {
    **cnn_test_probabilities,
    "Stacking Ensemble": stack_test_probabilities,
}


## 11. Test-set metrics and detailed evaluation

All reported final metrics below are computed on the untouched test split. Precision, recall, and F1 are reported macro-averaged, weighted-averaged, and per class. Specificity is computed one-vs-rest for each class. ROC and precision-recall curves are multiclass one-vs-rest curves, with macro and micro summaries where defined.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
import seaborn as sns

y_test_onehot = label_binarize(y_test, classes=np.arange(len(CLASS_NAMES)))
metric_rows = []
per_class_rows = []
evaluation_details = {}

for model_name, probabilities in all_test_probabilities.items():
    y_pred = np.argmax(probabilities, axis=1)
    matrix = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))
    specificity = {}
    for class_index, class_name in enumerate(CLASS_NAMES):
        true_positive = matrix[class_index, class_index]
        false_negative = matrix[class_index, :].sum() - true_positive
        false_positive = matrix[:, class_index].sum() - true_positive
        true_negative = matrix.sum() - true_positive - false_negative - false_positive
        specificity[class_name] = (
            float(true_negative / (true_negative + false_positive))
            if (true_negative + false_positive) else float("nan")
        )

    report = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    one_vs_rest_auc = {
        CLASS_NAMES[i]: float(roc_auc_score(y_test_onehot[:, i], probabilities[:, i]))
        for i in range(len(CLASS_NAMES))
    }
    one_vs_rest_ap = {
        CLASS_NAMES[i]: float(average_precision_score(y_test_onehot[:, i], probabilities[:, i]))
        for i in range(len(CLASS_NAMES))
    }
    try:
        macro_auc = float(roc_auc_score(
            y_test, probabilities, labels=np.arange(len(CLASS_NAMES)),
            multi_class="ovr", average="macro"
        ))
        weighted_auc = float(roc_auc_score(
            y_test, probabilities, labels=np.arange(len(CLASS_NAMES)),
            multi_class="ovr", average="weighted"
        ))
    except ValueError as exc:
        macro_auc = weighted_auc = float("nan")
        print(f"Could not calculate multiclass ROC-AUC for {model_name}: {exc}")

    try:
        macro_ap = float(average_precision_score(y_test_onehot, probabilities, average="macro"))
        micro_ap = float(average_precision_score(y_test_onehot, probabilities, average="micro"))
    except ValueError as exc:
        macro_ap = micro_ap = float("nan")
        print(f"Could not calculate average precision for {model_name}: {exc}")

    metric_rows.append({
        "model": model_name,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
        "roc_auc_ovr_macro": macro_auc,
        "roc_auc_ovr_weighted": weighted_auc,
        "average_precision_macro": macro_ap,
        "average_precision_micro": micro_ap,
        **{f"specificity_{name}": value for name, value in specificity.items()},
    })
    for class_name in CLASS_NAMES:
        per_class_rows.append({
            "model": model_name,
            "class": class_name,
            "precision": float(report[class_name]["precision"]),
            "recall": float(report[class_name]["recall"]),
            "f1_score": float(report[class_name]["f1-score"]),
            "support": int(report[class_name]["support"]),
            "specificity": specificity[class_name],
            "roc_auc_ovr": one_vs_rest_auc[class_name],
            "average_precision": one_vs_rest_ap[class_name],
        })
    evaluation_details[model_name] = {
        "y_pred": y_pred,
        "probabilities": probabilities,
        "confusion_matrix": matrix,
        "report": report,
        "specificity": specificity,
        "roc_auc_ovr_per_class": one_vs_rest_auc,
        "average_precision_per_class": one_vs_rest_ap,
    }

metrics_df = pd.DataFrame(metric_rows)
per_class_df = pd.DataFrame(per_class_rows)
display(metrics_df.round(4))
display(per_class_df.round(4))
metrics_df.to_csv(RUN_DIR / "test_metrics_summary.csv", index=False)
per_class_df.to_csv(RUN_DIR / "test_metrics_per_class.csv", index=False)

for model_name, details in evaluation_details.items():
    print(f"\n{model_name} — per-class classification report")
    print(classification_report(
        y_test,
        details["y_pred"],
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        zero_division=0,
    ))


## 12. Confusion matrices, ROC, and precision-recall curves

In the confusion matrix, rows are true classes and columns are predicted classes; off-diagonal cells show which classes are confused. ROC curves show the true-positive/false-positive trade-off for each one-vs-rest class; the dotted diagonal is chance level. Precision-recall curves show precision versus recall for each one-vs-rest class; average precision summarizes each curve and is useful when class counts differ.


In [ ]:
def save_confusion_plot(model_name, matrix, normalize=False):
    shown = matrix.astype(float)
    fmt = "d"
    if normalize:
        row_sums = shown.sum(axis=1, keepdims=True)
        shown = np.divide(shown, row_sums, out=np.zeros_like(shown), where=row_sums != 0)
        fmt = ".2f"
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        shown, annot=True, fmt=fmt, cmap="Blues", cbar=True, ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    )
    ax.set(title=f"{model_name}: {'normalized ' if normalize else ''}confusion matrix",
           xlabel="Predicted class", ylabel="True class")
    fig.tight_layout()
    suffix = "normalized" if normalize else "counts"
    fig.savefig(PLOT_DIR / f"{model_name.replace(' ', '_')}_confusion_{suffix}.png",
                dpi=160, bbox_inches="tight")
    plt.show()

for model_name, details in evaluation_details.items():
    save_confusion_plot(model_name, details["confusion_matrix"], normalize=False)
    save_confusion_plot(model_name, details["confusion_matrix"], normalize=True)


In [ ]:
for model_name, details in evaluation_details.items():
    probabilities = details["probabilities"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    class_fprs, class_tprs = [], []
    for class_index, class_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_test_onehot[:, class_index], probabilities[:, class_index])
        class_auc = auc(fpr, tpr)
        class_fprs.append(fpr)
        class_tprs.append(tpr)
        axes[0].plot(fpr, tpr, label=f"{class_name} (AUC={class_auc:.3f})")
    micro_fpr, micro_tpr, _ = roc_curve(y_test_onehot.ravel(), probabilities.ravel())
    axes[0].plot(
        micro_fpr, micro_tpr, linestyle=":", linewidth=2,
        label=f"Micro-average (AUC={auc(micro_fpr, micro_tpr):.3f})",
    )
    all_fpr = np.unique(np.concatenate(class_fprs))
    mean_tpr = np.zeros_like(all_fpr)
    for fpr, tpr in zip(class_fprs, class_tprs):
        mean_tpr += np.interp(all_fpr, fpr, tpr)
    mean_tpr /= len(CLASS_NAMES)
    axes[0].plot(
        all_fpr, mean_tpr, linestyle="--", linewidth=2,
        label=f"Macro-average (AUC={auc(all_fpr, mean_tpr):.3f})",
    )
    axes[0].plot([0, 1], [0, 1], "k--", label="Chance")
    axes[0].set(title=f"{model_name}: one-vs-rest ROC", xlabel="False positive rate",
                ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.02))
    axes[0].legend(loc="lower right")

    for class_index, class_name in enumerate(CLASS_NAMES):
        precision_curve, recall_curve, _ = precision_recall_curve(
            y_test_onehot[:, class_index], probabilities[:, class_index]
        )
        ap = details["average_precision_per_class"][class_name]
        axes[1].plot(recall_curve, precision_curve, label=f"{class_name} (AP={ap:.3f})")
    axes[1].set(title=f"{model_name}: one-vs-rest precision-recall",
                xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
    axes[1].legend(loc="best")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{model_name.replace(' ', '_')}_roc_pr.png",
                dpi=160, bbox_inches="tight")
    plt.show()


## 13. Compare test metrics and training time

Macro averages weight the three classes equally; weighted averages account for class support. Compare both so the larger Malignant test group does not hide weaker results on Benign or Normal. CNN training times include the full train/validation fit, not the separate OOF-generation cost.


In [ ]:
comparison_columns = [
    "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
]
comparison = metrics_df.set_index("model")[comparison_columns]
ax = comparison.plot(kind="bar", figsize=(14, 6), ylim=(0, 1), rot=20)
ax.set(title="Test-set metric comparison", ylabel="Score", xlabel="Model")
ax.legend(loc="lower right", ncol=2)
plt.tight_layout()
plt.savefig(PLOT_DIR / "test_metric_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

time_df = pd.DataFrame([
    {"model": name, "training_time_minutes": seconds / 60}
    for name, seconds in training_times.items()
])
ax = time_df.set_index("model")["training_time_minutes"].plot(kind="bar", figsize=(9, 5), color="teal")
ax.set(title="Full-training CNN time (OOF fits excluded)", ylabel="Minutes", xlabel="CNN")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(PLOT_DIR / "cnn_training_time_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
display(time_df)
time_df.to_csv(RUN_DIR / "cnn_training_times.csv", index=False)


## 14. Save predictions, models, and experiment configuration

The test predictions include true/predicted labels and all class probabilities. CNN best checkpoints are already saved in `checkpoints/`; the stacker is in the run folder. The JSON configuration includes the count check, environment versions, seeds, training parameters, OOF settings, and measured CNN training times.


In [ ]:
prediction_frame = pd.DataFrame({
    "image_path": test_paths,
    "true_label": [CLASS_NAMES[index] for index in y_test],
})
for model_name, details in evaluation_details.items():
    prediction_frame[f"{model_name}_predicted_label"] = [
        CLASS_NAMES[index] for index in details["y_pred"]
    ]
    for class_index, class_name in enumerate(CLASS_NAMES):
        prediction_frame[f"{model_name}_prob_{class_name}"] = details["probabilities"][:, class_index]
prediction_frame.to_csv(RUN_DIR / "test_predictions.csv", index=False)

configuration = {
    "run_id": RUN_ID,
    "dataset_root": str(DATASET_ROOT),
    "results_directory": str(RUN_DIR),
    "class_names_in_label_order": CLASS_NAMES,
    "expected_counts": EXPECTED_COUNTS,
    "observed_counts": {
        split: {
            class_name: len(split_files[split][class_name])
            for class_name in CLASS_NAMES
        }
        for split in ("train", "validation", "test")
    },
    "image_extensions": sorted(IMAGE_EXTENSIONS),
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "loss": "SparseCategoricalCrossentropy",
    "optimizer": "Adam",
    "backbones": MODEL_NAMES,
    "backbone_weights": "ImageNet",
    "vgg_head_mode": VGG_HEAD_MODE,
    "pipeline_cache": PIPELINE_CACHE,
    "seed": SEED,
    "tensorflow_deterministic_ops_enabled": DETERMINISM_ENABLED,
    "oof_folds": OOF_FOLDS,
    "oof_fit_policy": "Stratified folds on train only; fixed epochs selected from each full train/validation run.",
    "stacking": "DecisionTree + RandomForest + ExtraTrees with sklearn cross-validated predict_proba and LogisticRegression final estimator.",
    "training_times_seconds": training_times,
    "best_epochs": best_epochs,
    "environment": ENVIRONMENT,
    "reproducibility_limitations": [
        "GPU kernels and hardware/library versions may cause numeric variation.",
        "ImageNet weight downloads and external runtime state may vary.",
        "Deterministic TensorFlow operations are enabled when supported, but exact cross-platform reproducibility is not guaranteed.",
        "OOF CNN features depend on fold membership and fixed fold-training epochs.",
    ],
}
with (RUN_DIR / "experiment_config.json").open("w") as file:
    json.dump(configuration, file, indent=2)

metrics_json = {
    "test_summary": metrics_df.replace({np.nan: None}).to_dict(orient="records"),
    "test_per_class": per_class_df.replace({np.nan: None}).to_dict(orient="records"),
}
with (RUN_DIR / "test_metrics.json").open("w") as file:
    json.dump(metrics_json, file, indent=2)

print("Saved experiment artifacts to:", RUN_DIR)
print("Main artifacts: checkpoints/, histories/, plots/, test_metrics.csv/json, test_predictions.csv, training_oof_features.csv/npz, stacking_model.joblib, experiment_config.json")


## 15. Optional Grad-CAM explanations

Grad-CAM highlights image regions that influence a selected CNN class score. These maps can be unstable and sensitive to preprocessing, model architecture, and image choice. They are exploratory visualizations only; they do **not** prove that the model uses medically valid evidence or establish clinical validity.


In [ ]:
if ENABLE_GRAD_CAM:
    def read_one_image(path, model_name):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
        raw_float = tf.cast(image, tf.float32)
        model_input = PREPROCESSORS[model_name](raw_float)
        return raw_float.numpy().astype(np.uint8), model_input

    def grad_cam(model, model_name, model_input, class_index):
        backbone = model.get_layer("feature_extractor")
        conv_layers = [
            layer for layer in backbone.layers
            if isinstance(layer, tf.keras.layers.Conv2D)
        ]
        if not conv_layers:
            raise ValueError(f"No convolutional layer found for {model_name}.")
        last_conv = conv_layers[-1]
        grad_model = tf.keras.Model(
            backbone.input,
            [last_conv.output, backbone.output],
        )
        with tf.GradientTape() as tape:
            conv_outputs, backbone_features = grad_model(
                tf.expand_dims(model_input, 0), training=False
            )
            x = backbone_features
            for layer in model.layers:
                if layer is backbone or isinstance(layer, tf.keras.layers.InputLayer):
                    continue
                x = layer(x, training=False) if isinstance(
                    layer, (tf.keras.layers.Dropout, tf.keras.layers.BatchNormalization)
                ) else layer(x)
            predictions = x
            class_score = predictions[:, class_index]
        gradients = tape.gradient(class_score, conv_outputs)
        weights = tf.reduce_mean(gradients, axis=(0, 1, 2))
        heatmap = tf.reduce_sum(conv_outputs[0] * weights, axis=-1)
        heatmap = tf.maximum(heatmap, 0)
        maximum = tf.reduce_max(heatmap)
        return (heatmap / maximum).numpy() if maximum > 0 else heatmap.numpy()

    representative_indices = [
        int(np.flatnonzero(y_test == class_index)[0])
        for class_index in range(len(CLASS_NAMES))
        if np.any(y_test == class_index)
    ]
    for model_name in MODEL_NAMES:
        cnn_model = tf.keras.models.load_model(model_paths[model_name])
        fig, axes = plt.subplots(len(representative_indices), 2, figsize=(8, 4 * len(representative_indices)))
        if len(representative_indices) == 1:
            axes = np.array([axes])
        for row, sample_index in enumerate(representative_indices):
            original, model_input = read_one_image(test_paths[sample_index], model_name)
            predicted_class = int(np.argmax(cnn_model(tf.expand_dims(model_input, 0), training=False)[0]))
            heatmap = grad_cam(cnn_model, model_name, model_input, predicted_class)
            heatmap_resized = tf.image.resize(heatmap[..., None], IMAGE_SIZE).numpy().squeeze()
            axes[row, 0].imshow(original)
            axes[row, 0].set_title(f"True: {CLASS_NAMES[y_test[sample_index]]}")
            axes[row, 1].imshow(original)
            axes[row, 1].imshow(heatmap_resized, cmap="jet", alpha=0.4)
            axes[row, 1].set_title(f"{model_name} Grad-CAM: {CLASS_NAMES[predicted_class]}")
            axes[row, 0].axis("off")
            axes[row, 1].axis("off")
        fig.tight_layout()
        fig.savefig(PLOT_DIR / f"{model_name}_grad_cam.png", dpi=160, bbox_inches="tight")
        plt.show()
        del cnn_model
        tf.keras.backend.clear_session()
else:
    print("Grad-CAM is disabled. Set ENABLE_GRAD_CAM = True and rerun this section to create example overlays.")


## 16. Optional SHAP explanations

SHAP is disabled by default because pixel-level explanations for large CNNs can be computationally expensive and can require an optional `shap` install. When available, this section uses `GradientExplainer` with a very small background for multiclass CNN outputs, and Kernel SHAP on the 12 probability features for the stacker. These are exploratory explanations, not causal evidence or clinical validation.


In [ ]:
if ENABLE_SHAP:
    try:
        import shap
    except ImportError as exc:
        raise ImportError(
            "SHAP is optional. Install shap in the notebook environment, then rerun this section."
        ) from exc

    # Use the first available test images; keep these sample sizes small due to runtime and memory.
    explain_indices = np.arange(min(SHAP_BACKGROUND_SIZE + SHAP_EXPLAIN_SIZE, len(test_paths)))
    if len(explain_indices) <= SHAP_BACKGROUND_SIZE:
        raise ValueError("Not enough test images for the configured SHAP background and explain sizes.")

    for model_name in MODEL_NAMES:
        cnn_model = tf.keras.models.load_model(model_paths[model_name])
        transformed = []
        for index in explain_indices:
            _, model_input = read_one_image(test_paths[index], model_name)
            transformed.append(model_input.numpy())
        transformed = np.asarray(transformed, dtype=np.float32)
        background = transformed[:SHAP_BACKGROUND_SIZE]
        explain_batch = transformed[SHAP_BACKGROUND_SIZE:]
        explainer = shap.GradientExplainer(cnn_model, background)
        shap_values = explainer.shap_values(explain_batch)
        print(f"{model_name}: multiclass SHAP returned {len(shap_values) if isinstance(shap_values, list) else 'array'} output(s).")
        # Save raw values for later inspection; plotting conventions vary by SHAP release.
        np.savez_compressed(
            RUN_DIR / f"{model_name}_shap_values.npz",
            values=np.asarray(shap_values, dtype=object),
            explained_indices=explain_indices[SHAP_BACKGROUND_SIZE:],
        )
        del cnn_model
        tf.keras.backend.clear_session()

    stack_background = test_feature_matrix[:min(SHAP_BACKGROUND_SIZE, len(test_feature_matrix))]
    stack_explain = test_feature_matrix[:min(SHAP_EXPLAIN_SIZE, len(test_feature_matrix))]
    stack_explainer = shap.KernelExplainer(stacking_model.predict_proba, stack_background)
    stack_shap_values = stack_explainer.shap_values(stack_explain, nsamples=100)
    np.savez_compressed(
        RUN_DIR / "stacking_model_shap_values.npz",
        values=np.asarray(stack_shap_values, dtype=object),
        feature_names=np.asarray(oof_feature_names, dtype=object),
    )
    print("Saved optional SHAP arrays. Interpret them cautiously; they are not clinical validation.")
else:
    print("SHAP is disabled. Set ENABLE_SHAP = True to attempt the small-sample optional workflow.")


## 17. Final checklist and limitations

- Verify the count table above; the notebook reports any difference from the expected split/class counts and never edits the dataset.
- Check that GPU memory can support the selected batch size and VGG head. If memory is insufficient, keep `VGG_HEAD_MODE = "gap"` and lower `BATCH_SIZE`.
- The full workflow includes four full CNN fits plus 20 fold CNN fits for five-fold OOF features; plan runtime accordingly.
- The validation set guides CNN early stopping/checkpoint selection. The stacking models are fitted using training-only OOF features. The test set is used only for final evaluation and optional illustrative explanations.
- This notebook does not reproduce an unspecified exact paper stacking protocol and does not establish clinical validity.

All outputs for this run are under:

```text
RUN_DIR (the timestamped run folder printed above)
```
